# Prototype Analysis on Sample

Dieses Notebook testet erste Analyseideen auf einem kleinen Sample der bereinigten Parking-Violations-Daten.

## Ziel

- processed Parquet-Daten aus HDFS laden
- kleines 1%-Sample erstellen
- Analysefragen testen
- häufigste Violation Codes untersuchen
- häufigste Vehicle Makes untersuchen
- zeitliche Muster nach Monat, Wochentag und Tageszeit prüfen
- beurteilen, welche Queries und Charts für die finale Analyse sinnvoll sind

Die finalen Analysen werden später in `src/4_Analysis` auf dem vollständigen Datensatz ausgeführt.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc

spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_Prototype") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memory", "15g") \
    .config("spark.cores.max", "12") \
    .getOrCreate()

spark


In [ ]:
processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned_v5"

df = spark.read.parquet(processed_path)

df.groupBy("fy").count().orderBy("fy").show()

In [ ]:
sample_df = df.sample(fraction=0.01, seed=42)

sample_count = sample_df.count()
sample_count

In [ ]:
sample_df.groupBy("violation_code", "violation_description_official") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)


In [ ]:
sample_df.groupBy("violation_code") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)

In [ ]:
sample_df.groupBy("vehicle_make") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)

In [ ]:
sample_df.groupBy("fy", "issue_month") \
    .count() \
    .orderBy("fy", "issue_month") \
    .show(50)

In [ ]:
sample_df.groupBy("issue_weekday") \
    .count() \
    .orderBy("issue_weekday") \
    .show()

In [ ]:
sample_df.filter(col("violation_hour").isNotNull()) \
    .groupBy("violation_hour") \
    .count() \
    .orderBy("violation_hour") \
    .show(24)

## Erkenntnisse aus dem Prototyping

Das 1%-Sample enthält 500'468 Zeilen und ist damit gross genug, um Analyseideen zu testen.

Getestete Analysefragen:

1. **Welche Violation Codes kommen am häufigsten vor?**  
   `violation_code = 36` (PHTO SCHOOL ZN SPEED VIOLATION) ist im Sample mit Abstand am häufigsten. Für die finale Analyse wird `violation_description_official` verwendet, da diese eine eindeutige offizielle Beschreibung pro Code liefert.

2. **Welche Vehicle Makes erhalten am häufigsten Parking Violations?**  
   Im Sample sind `HONDA`, `TOYOT`, `FORD` und `NISSA` besonders häufig.

3. **Gibt es zeitliche Muster nach Monat?**  
   Besonders FY2023 zeigt auffällige Werte in den Monaten Juli, August und September. Bei FY2024 sind Juli und August kaum vertreten, da Violations aus diesen Monaten im Pre-processing als Duplikate mit FY2023 entfernt wurden. September 2024 zeigt ebenfalls wenige Einträge — eine mögliche Erklärung wäre eine Datenlücke im Originaldatensatz.

4. **Gibt es zeitliche Muster nach Wochentag?**  
   Sonntag (Wochentag 1) weist deutlich weniger Verstösse auf als die Werktage.

5. **Gibt es zeitliche Muster nach Tageszeit?**  
   Im Sample zeigen sich hohe Werte insbesondere zwischen ca. 08:00 und 14:00 Uhr. Für diese Analyse werden nur Datensätze mit `violation_hour IS NOT NULL` verwendet.

Die getesteten Analysen werden im nächsten Schritt in `src/4_Analysis` auf dem vollständigen Datensatz ausgeführt.

In [ ]:
spark.stop()